In [32]:
import numpy as np
import pandas as pd
from ydata_profiling import ProfileReport
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from autogluon.tabular import TabularPredictor
import plotly.express as px
from pprint import pprint
from autogluon.features.generators import AutoMLPipelineFeatureGenerator

In [33]:
csv_path = Path.cwd().parents[1] / "exports" / "survey_2025_scalar_columns.csv"
df = pd.read_csv(csv_path, low_memory=False)

In [34]:
pd.options.display.max_columns = 500

df

,ResponseId,MainBranch,Age,EdLevel,Employment,WorkExp,LearnCodeChoose,LearnCodeAI,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,Industry,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,Currency,CompTotal,LanguageChoice,DatabaseChoice,PlatformChoice,WebframeChoice,DevEnvsChoice,AIModelsChoice,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIAgents,AIAgentChange,ConvertedCompYearly,JobSat
0,1,I am a developer by profession,25-34 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,8.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",14.0,"Developer, mobile",20 to 99 employees,People manager,Remote,"Yes, I influenced the purchase of a substantial addition to the tech stack",Work,Fintech,I'm not sure,I have neither consider or transitioned into a new career or industry,7.0,3.0,Ukraine,EUR European Euro,52800.0,Yes,Yes,Yes,No,Yes,Yes,Yes,A few times per week,Between 5 and 10 years,I have never participated in Q&A on Stack Overflow,Neutral,"Rarely, almost never","Yes, I use AI tools monthly or infrequently",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,"Yes, I use AI agents at work monthly or infrequently",Not at all or minimally,61256.0,10.0
1,2,I am a developer by profession,25-34 years old,"Associate degree (A.A., A.S., etc.)",Employed,2.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",10.0,"Developer, back-end",500 to 999 employees,Individual contributor,"Hybrid (some in-person, leans heavy to flexibility)",No,Personal Project,Retail and Consumer Services,I'm not sure,I have transitioned into a new career and/or industry voluntarily,6.0,5.0,Netherlands,EUR European Euro,90000.0,Yes,Yes,Yes,Yes,Yes,Yes,Not sure/can't remember,Multiple times per day,Between 10 and 15 years,"Infrequently, less than once per year","Yes, somewhat",About half of the time,"Yes, I use AI tools weekly",Indifferent,Neither trust nor distrust,Bad at handling complex tasks,"No, and I don't plan to",Not at all or minimally,104413.0,9.0
2,3,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)","Independent contractor, freelancer, or self-employed",10.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",12.0,"Developer, front-end",NaN,NaN,NaN,No,Work,Software Development,No,I have transitioned into a new career and/or industry involuntarily,3.0,3.0,Ukraine,UAH Ukrainian hryvnia,2214000.0,Yes,Yes,Yes,Yes,Yes,Yes,Not sure/can't remember,A few times per week,Between 5 and 10 years,"Infrequently, less than once per year",Neutral,About half of the time,"Yes, I use AI tools daily",Favorable,Somewhat trust,Neither good or bad at handling complex tasks,"Yes, I use AI agents at work weekly","Yes, somewhat",53061.0,8.0
3,4,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,4.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",5.0,"Developer, back-end","10,000 or more employees",Individual contributor,Remote,No,Personal Project,Retail and Consumer Services,No,I have neither consider or transitioned into a new career or industry,NaN,NaN,Ukraine,EUR European Euro,31200.0,Yes,No,Yes,Yes,No,No,Yes,A few times per month or weekly,Between 3 and 5 years,I have never participated in Q&A on Stack Overflow,Neutral,"Rarely, almost never","Yes, I use AI tools weekly",Favorable,Somewhat trust,Bad at handling complex tasks,"Yes, I use AI agents at work monthly or 

In [ ]:
# Auto EDA
# profile = ProfileReport(df, title="Stack Overflow Developer Survey 2025 - Public Results (Scalars)")
# profile.to_file("so_survey_2025_profile_scalars_report.html")
# profile.to_file("so_survey_2025_profile_scalars_report.json")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 227.48it/s]


### Baseline with AutoGluon Tabular: predict `ConvertedCompYearly`

**Goal:** A strong default model with minimal manual feature engineering.

**How AutoGluon Tabular works:** We pass one dataframe with a **label** column. On `fit`, it (1) runs a **feature pipeline** (dtype inference, categorical encoding, and optional text/datetime handling), (2) **fits many models** (e.g. gradient boosting, random forest, neural nets) using **internal** train/validation splits of the training frame you pass in, (3) picks and often **ensembles** winners by validation score. It does **not** remove **label leakage** for us, anything that is effectively part of the target definition must be dropped **before** `fit`.

- **Target filter** — `ConvertedCompYearly` is continuous compensation in USD. Rows with missing or non-positive targets are dropped so supervised training and metrics are well-defined.
- **Drop `LEAK_COLS`** — `ConvertedCompYearly` is derived from `CompTotal`, `Currency`, and Stack Overflow’s conversion rules. Using those as features would **leak** the answer and make test scores meaningless. `ResponseId` is only an identifier, not a causal input.
- **`train_test_split` (80/20)** — This is an **outer** test set: never passed to `fit`, used only for final evaluation. AutoGluon will **again** split `train_data` inside `fit` for model selection and stacking; that inner split is not a replacement for this holdout.


In [35]:
# Regression label: annual compensation in USD
TARGET = "ConvertedCompYearly"
# Do not use as features — CompTotal/Currency are inputs to the same conversion as the label; ResponseId is not predictive.
LEAK_COLS = ["ResponseId", "CompTotal", "Currency"]

# Supervised rows only: need a real positive salary for training and sensible metrics.
df = df[df[TARGET].notna() & (df[TARGET] > 0)].reset_index(drop=True)
df = df.drop(columns=[c for c in LEAK_COLS if c in df.columns])

# Outer holdout for honest test metrics. TabularPredictor.fit() still splits this train set internally.
train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42, shuffle=True
)

print(f"Rows: {len(df):,}  |  Train: {len(train_data):,}  |  Test: {len(test_data):,}")
train_data.head()

Rows: 23,947  |  Train: 19,157  |  Test: 4,790


,MainBranch,Age,EdLevel,Employment,WorkExp,LearnCodeChoose,LearnCodeAI,YearsCode,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,Industry,AIThreat,NewRole,ToolCountWork,ToolCountPersonal,Country,LanguageChoice,DatabaseChoice,PlatformChoice,WebframeChoice,DevEnvsChoice,AIModelsChoice,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIAgents,AIAgentChange,ConvertedCompYearly,JobSat
18879,I am a developer by profession,35-44 years old,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",Employed,3.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools required for my job or to benefit my career",5.0,Data engineer,100 to 499 employees,Individual contributor,"Hybrid (some in-person, leans heavy to flexibility)",No,Work,Software Development,No,I have transitioned into a new career and/or industry voluntarily,3.0,0.0,United Kingdom of Great Britain and Northern Ireland,Yes,Yes,Yes,No,Yes,Yes,Yes,A few times per month or weekly,Between 3 and 5 years,"Infrequently, less than once per year","No, not really","Rarely, almost never","Yes, I use AI tools daily",Very favorable,Somewhat trust,I don't use AI tools for complex tasks / I don't know,"No, I use AI exclusively in copilot/autocomplete mode",Not at all or minimally,59962.0,10.0
3162,I am a developer by profession,25-34 years old,"Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)",Employed,6.0,"No, I am not new to coding and did not learn new coding techniques or programming languages","No, I didn't spend time learning in the past year",6.0,"Developer, front-end",20 to 99 employees,Individual contributor,Remote,No,Personal Project,Higher Education,Yes,I have somewhat considered changing my career and/or the industry I work in,13.0,5.0,Germany,Yes,Yes,Yes,Yes,Yes,NaN,No,A few times per month or weekly,Between 5 and 10 years,I have never participated in Q&A on Stack Overflow,"No, not at all","Rarely, almost never","Yes, I use AI tools daily",Favorable,Somewhat distrust,Very poor at handling complex tasks,"Yes, I use AI agents at work monthly or infrequently","Yes, to a great extent",81210.0,8.0
15568,I am a developer by profession,35-44 years old,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Employed,13.0,"Yes, I am not new to coding but am learning new coding techniques or programming language","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",19.0,"Developer, back-end","10,000 or more employees",Individual contributor,"Your choice (very flexible, you can come in when you want or just as needed)",No,Work,Fintech,No,I have somewhat considered changing my career and/or the industry I work in,10.0,NaN,Romania,Yes,Yes,Yes,No,Yes,Yes,Yes,A few times per week,Between 10 and 15 years,Less than once every 2 - 3 months,"Yes, somewhat","Rarely, almost never","Yes, I use AI tools monthly or infrequently",Favorable,Neither trust nor distrust,Neither good or bad at handling complex tasks,"No, I use AI exclusively in copilot/autocomplete mode","Yes, somewhat",88171.0,8.0
1731,I am a developer by profession,35-44 years old,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Employed,19.0,"No, I am not new to coding and did not learn new coding techniques or programming languages","Yes, I learned how to use AI-enabled tools for my personal curiosity and/or hobbies",19.0,"Developer, full-stack",Less than 20 employees,Individual contributor,Remote,No,Work,Software Development,I'm not sure,I have somewhat considered changing my career and/or the industry I work in,20.0,0.0,United States of America,Yes,Yes,Yes,Yes,Yes,No,Yes,A few times per month or weekly,Between 10 and 15 years,I have never participated in Q&A on Stack Overflow,"No, not at all","Rarely, almost never","Yes, I use AI tools daily",Favorable,Somewhat trust,"Good, but not great at handling complex tasks","No, but I plan to","Yes, so

In [58]:
fig = px.histogram(
    train_data,
    x=TARGET,
    nbins=60,
    title="Distribution of ConvertedCompYearly - training set (baseline)",
    labels={TARGET: "Salary ($)"},
)
fig.update_layout(showlegend=False)
fig.show()

In [37]:
X = train_data.drop(columns=[TARGET])  # or your label name
fg = AutoMLPipelineFeatureGenerator()
X_processed = fg.fit_transform(X)
X_processed.head()

Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    8567.30 MB
	Train Data (Original)  Memory Usage: 46.59 MB (0.5% of available memory)


	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])  :  5 | ['WorkExp', 'YearsCode', 'ToolCountWork', 'ToolCountPersonal', 'JobSat']
		('object', []) : 34 | ['MainBranch', 'Age', 'EdLevel', 'Employment', 'LearnCodeChoose', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('category', []) : 34 | ['MainBranch', 'Age', 'EdLevel', 'Employment', 'LearnCodeChoose', ...]
		('float', [])    :  5 | ['WorkExp', 'YearsCode', 'T

,WorkExp,YearsCode,ToolCountWork,ToolCountPersonal,JobSat,MainBranch,Age,EdLevel,Employment,LearnCodeChoose,LearnCodeAI,DevType,OrgSize,ICorPM,RemoteWork,PurchaseInfluence,TechEndorseIntro,Industry,AIThreat,NewRole,Country,LanguageChoice,DatabaseChoice,PlatformChoice,WebframeChoice,DevEnvsChoice,AIModelsChoice,SOAccount,SOVisitFreq,SODuration,SOPartFreq,SOComm,SOFriction,AISelect,AISent,AIAcc,AIComplex,AIAgents,AIAgentChange
18879,3.0,5.0,3.0,0.0,10.0,0,2,6,0,3,5,6,3,1,1,1,3,14,2,5,132,2,2,2,1,2,2,3,1,3,5,3,5,3,5,5,3,1,3
3162,6.0,6.0,13.0,5.0,8.0,0,1,7,0,1,1,16,4,1,4,1,1,7,3,2,42,2,2,2,2,2,0,1,1,4,4,2,5,3,1,4,5,5,5
15568,13.0,19.0,10.0,NaN,8.0,0,2,3,0,3,4,13,2,1,5,1,3,4,2,2,104,2,2,2,1,2,2,3,2,2,6,6,5,4,1,3,4,1,4
1731,19.0,19.0,20.0,0.0,9.0,0,2,2,0,1,4,17,9,1,4,1,3,14,1,2,134,2,2,2,2,2,1,3,1,2,4,2,5,3,1,5,2,3,4
16899,1.0,2.0,5.0,2.0,NaN,1,0,2,0,3,4,7,6,1,2,1,1,9,3,3,51,2,2,1,1,2,1,3,1,6,4,6,5,2,5,5,3,3,4


In [59]:
fig = px.histogram(
    X_processed,
    x="WorkExp",
    nbins=60,
    title="Distribution of WorkExp - training set (baseline)",
    labels={"WorkExp": "Years of Work Experience"},
)
fig.update_layout(showlegend=False)
fig.show()

In [39]:
predictor = TabularPredictor.load(
    str((Path.cwd() / "AutogluonModels" / "ag-20260414_160116").resolve())
)

# To retrain from scratch, uncomment below and comment out the load above:
# predictor = TabularPredictor(
#     label=TARGET, problem_type="regression", eval_metric="mean_absolute_error",
# ).fit(train_data=train_data, time_limit=10 * 60, presets="best_quality")

In [40]:
predictor.leaderboard(test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,-40998.822516,-42184.207380,mean_absolute_error,1.761408,2.268790,385.341789,0.001797,0.000555,0.058814,3,True,12
1,CatBoost_BAG_L2,-41021.553592,-42201.266399,mean_absolute_error,1.759611,2.268235,385.282975,0.078622,0.152113,103.522172,2,True,9
2,WeightedEnsemble_L2,-41107.096895,-42315.538961,mean_absolute_error,0.449632,0.355728,253.844382,0.001448,0.000442,0.048580,2,True,7
3,CatBoost_BAG_L1,-41176.018443,-42393.893980,mean_absolute_error,0.151765,0.177360,229.936886,0.151765,0.177360,229.936886,1,True,2
4,CatBoost_r177_BAG_L1,-41920.638251,-43156.812047,mean_absolute_error,0.026556,0.105560,2.884699,0.026556,0.105560,2.884699,1,True,6
5,NeuralNetTorch_BAG_L1,-44065.934311,-46798.599044,mean_absolute_error,0.296419,0.177926,23.858916,0.296419,0.177926,23.858916,1,True,5
6,NeuralNetFastAI_BAG_L1,-48499.372616,-55925.827745,mean_absolute_error,0.410740,0.251136,14.832108,0.410740,0.251136,14.832108,1,True,4
7,NeuralNetFastAI_BAG_L2,-48911.938215,-50307.325340,mean_absolute_error,2.069277,2.351177,295.762796,0.388288,0.235056,14.001993,2,True,11
8,RandomForestMSE_BAG_L2,-51178.595717,-58102.654064,mean_absolute_error,2.011616,2.836947,295.445206,0.330627,0.720825,13.684403,2,True,8
9,ExtraTreesMSE_BAG_L2,-54966.319962,-58304.702928,mean_absolute_error,1.894538,2.888604,285.828236,0.213549,0.772482,4.067433,2,True,10


In [41]:
predictor.evaluate(test_data)


{'mean_absolute_error': -40998.8225163629,
 'root_mean_squared_error': np.float64(-231776.2931159863),
 'mean_squared_error': -53720250050.5876,
 'r2': 0.04946239768736671,
 'pearsonr': 0.23479075157718104,
 'median_absolute_error': -17194.935546875}

In [42]:
predictor.model_best

'WeightedEnsemble_L3'

In [43]:
baseline_preds = predictor.predict(test_data)
baseline_actual = test_data[TARGET]

fig = px.scatter(
    x=baseline_actual,
    y=baseline_preds,
    labels={"x": "Actual Salary ($)", "y": "Predicted Salary ($)"},
    title="Baseline: Actual vs Predicted",
    opacity=0.3,
)
fig.add_shape(
    type="line", x0=0, y0=0,
    x1=baseline_actual.max(), y1=baseline_actual.max(),
    line=dict(dash="dash", color="red"),
)
fig.update_layout(showlegend=False)
fig.show()

In [45]:
baseline_residuals = baseline_actual - baseline_preds

fig = px.histogram(
    baseline_residuals,
    nbins=80,
    title="Baseline: Residuals (Actual - Predicted)",
    labels={"value": "Residual ($)"},
)
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(showlegend=False)
fig.show()

In [23]:
predictor.feature_importance(test_data)

Computing feature importance via permutation shuffling for 39 features using 4790 rows with 5 shuffle sets...
	300.97s	= Expected runtime (60.19s per shuffle set)
	152.33s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
Country,24364.608881,324.217534,3.761724e-09,5,25032.177265,23697.040497
WorkExp,4017.417273,100.829964,4.757593e-08,5,4225.027561,3809.806984
OrgSize,1770.163181,152.446338,6.536022e-06,5,2084.052297,1456.274064
DevType,1222.945905,49.561489,3.229827e-07,5,1324.993696,1120.898115
Industry,821.077637,65.793022,4.905178e-06,5,956.546379,685.608895
Age,820.545173,110.627401,3.870519e-05,5,1048.328521,592.761825
YearsCode,587.897890,111.498074,1.480801e-04,5,817.473966,358.321813
Employment,517.900592,45.416709,7.024546e-06,5,611.414224,424.386961
RemoteWork,385.788589,36.683502,9.692802e-06,5,461.320427,310.256751
EdLevel,311.768129,49.093777,7.140626e-05,5,412.852894,210.683364


### Cleaned + Log-Target Model

**Motivation:** The baseline trains on raw data with minimal filtering. Two changes should improve performance:

1. **Data cleaning** (ported from the EDA salary analysis notebook) — keep only professional developers, remove rows with inconsistent age/experience fields, and cap compensation to \$1,000–\$500,000 to exclude non-meaningful entries and extreme outliers.
2. **Log-transformed target** — `ConvertedCompYearly` is heavily right-skewed. Predicting `log10(salary)` lets the model focus on *relative* differences (a \$10k error matters more at \$30k than at \$300k) and produces a more symmetric loss landscape.

We evaluate in **both** log-space and dollar-space so results are directly comparable to the baseline (~\$41k MAE).

In [46]:
df_clean = df.copy()
n_start = len(df_clean)
cleaning_log = []

def log_step(step, n_before, n_after):
    cleaning_log.append({"step": step, "removed": n_before - n_after, "remaining": n_after})

# 1a. Professional developers only
n_before = len(df_clean)
df_clean = df_clean[df_clean["MainBranch"] == "I am a developer by profession"]
log_step("Keep professional developers only", n_before, len(df_clean))

# 1b. Must have compensation
n_before = len(df_clean)
df_clean = df_clean.dropna(subset=["ConvertedCompYearly"])
log_step("Drop missing ConvertedCompYearly", n_before, len(df_clean))

# 2. Age / experience consistency
AGE_UPPER_BOUND = {
    "18-24 years old": 24,
    "25-34 years old": 34,
    "35-44 years old": 44,
    "45-54 years old": 54,
    "55-64 years old": 64,
    "65 years or older": 80,
    "Prefer not to say": np.nan,
}
df_clean["_age_upper"] = df_clean["Age"].map(AGE_UPPER_BOUND)

n_before = len(df_clean)
mask = df_clean["_age_upper"].notna() & df_clean["WorkExp"].notna()
df_clean = df_clean[~(mask & (df_clean["WorkExp"] > df_clean["_age_upper"]))]
log_step("WorkExp > Age upper bound", n_before, len(df_clean))

n_before = len(df_clean)
mask = df_clean["_age_upper"].notna() & df_clean["YearsCode"].notna()
df_clean = df_clean[~(mask & (df_clean["YearsCode"] > df_clean["_age_upper"]))]
log_step("YearsCode > Age upper bound", n_before, len(df_clean))

n_before = len(df_clean)
mask = df_clean["WorkExp"].notna() & df_clean["YearsCode"].notna()
df_clean = df_clean[~(mask & (df_clean["WorkExp"] > df_clean["YearsCode"] + 5))]
log_step("WorkExp > YearsCode + 5 (suspicious gap)", n_before, len(df_clean))

df_clean = df_clean.drop(columns=["_age_upper"])

# 3. Compensation bounds
COMP_MIN = 1_000
COMP_MAX = 500_000

n_before = len(df_clean)
df_clean = df_clean[df_clean["ConvertedCompYearly"] >= COMP_MIN]
log_step(f"ConvertedCompYearly < ${COMP_MIN:,}", n_before, len(df_clean))

n_before = len(df_clean)
df_clean = df_clean[df_clean["ConvertedCompYearly"] <= COMP_MAX]
log_step(f"ConvertedCompYearly > ${COMP_MAX:,}", n_before, len(df_clean))

summary = pd.DataFrame(cleaning_log)
summary.loc[len(summary)] = {"step": "TOTAL REMOVED", "removed": n_start - len(df_clean), "remaining": len(df_clean)}
summary

,step,removed,remaining
0,Keep professional developers only,3752,20195
1,Drop missing ConvertedCompYearly,0,20195
2,WorkExp > Age upper bound,8,20187
3,YearsCode > Age upper bound,4,20183
4,WorkExp > YearsCode + 5 (suspicious gap),325,19858
5,"ConvertedCompYearly < $1,000",456,19402
6,"ConvertedCompYearly > $500,000",147,19255
7,TOTAL REMOVED,4692,19255


In [47]:
TARGET = "ConvertedCompYearly"
LOG_TARGET = "LogCompYearly"
LEAK_COLS = ["ResponseId", "CompTotal", "Currency"]

df_clean = df_clean.drop(columns=[c for c in LEAK_COLS if c in df_clean.columns])
df_clean[LOG_TARGET] = np.log10(df_clean[TARGET])
df_clean = df_clean.drop(columns=[TARGET])
df_clean = df_clean.reset_index(drop=True)

train_clean, test_clean = train_test_split(
    df_clean, test_size=0.2, random_state=42, shuffle=True
)

print(f"Rows: {len(df_clean):,}  |  Train: {len(train_clean):,}  |  Test: {len(test_clean):,}")

Rows: 19,255  |  Train: 15,404  |  Test: 3,851


In [48]:
fig = px.histogram(
    10 ** train_clean[LOG_TARGET],
    nbins=60,
    title="Distribution of ConvertedCompYearly - training set",
    labels={"value": "Salary ($)"},
)
fig.update_layout(showlegend=False)
fig.show()

In [49]:
fig = px.histogram(
    train_clean,
    x=LOG_TARGET,
    nbins=60,
    title="Distribution of log10(ConvertedCompYearly) — training set",
    labels={LOG_TARGET: "log₁₀(Salary)"},
)
fig.show()

In [50]:
predictor_v2 = TabularPredictor.load(
    str((Path.cwd() / "AutogluonModels" / "ag-20260521_144442").resolve())
)

# To retrain from scratch, uncomment below and comment out the load above:
# predictor_v2 = TabularPredictor(
#     label=LOG_TARGET, problem_type="regression", eval_metric="mean_absolute_error",
# ).fit(train_data=train_clean, time_limit=10 * 60, presets="best_quality")

In [51]:
log_metrics = predictor_v2.evaluate(test_clean)
print("=== Log-space metrics (AutoGluon) ===")
pprint(log_metrics)

# Back-transform to dollar space for apples-to-apples comparison with baseline
preds_log = predictor_v2.predict(test_clean)
preds_dollars = 10 ** preds_log
actual_dollars = 10 ** test_clean[LOG_TARGET]

dollar_mae = mean_absolute_error(actual_dollars, preds_dollars)
dollar_rmse = root_mean_squared_error(actual_dollars, preds_dollars)
dollar_r2 = r2_score(actual_dollars, preds_dollars)
dollar_median_ae = np.median(np.abs(actual_dollars - preds_dollars))

print("\n=== Dollar-space metrics (back-transformed) ===")
print(f"  MAE:       ${dollar_mae:,.0f}")
print(f"  MedAE:     ${dollar_median_ae:,.0f}")
print(f"  RMSE:      ${dollar_rmse:,.0f}")
print(f"  R²:        {dollar_r2:.4f}")
print(f"\n  (Baseline MAE was ~$41,000)")

=== Log-space metrics (AutoGluon) ===
{'mean_absolute_error': -0.17660087428494034,
 'mean_squared_error': -0.08107517448791168,
 'median_absolute_error': -0.10802030329286172,
 'pearsonr': 0.7809657804587935,
 'r2': 0.603158880714588,
 'root_mean_squared_error': np.float64(-0.284737026900106)}

=== Dollar-space metrics (back-transformed) ===
  MAE:       $28,671
  MedAE:     $16,564
  RMSE:      $47,826
  R²:        0.5524

  (Baseline MAE was ~$41,000)


In [52]:
predictor_v2.leaderboard(test_clean)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetTorch_r79_BAG_L2,-0.176455,-0.178594,mean_absolute_error,1.598740,2.506825,292.533510,0.324827,0.249517,10.824332,2,True,14
1,WeightedEnsemble_L3,-0.176601,-0.175561,mean_absolute_error,2.361867,3.210857,397.334805,0.001685,0.000772,0.107570,3,True,15
2,CatBoost_BAG_L2,-0.177777,-0.177418,mean_absolute_error,1.345853,2.404536,353.953231,0.071940,0.147229,72.244053,2,True,9
3,CatBoost_r177_BAG_L2,-0.178165,-0.177698,mean_absolute_error,1.296098,2.323610,288.288051,0.022185,0.066302,6.578873,2,True,13
4,WeightedEnsemble_L2,-0.178774,-0.177233,mean_absolute_error,0.796575,1.050193,273.043009,0.001722,0.000508,0.062763,2,True,7
5,NeuralNetTorch_BAG_L2,-0.179599,-0.179379,mean_absolute_error,1.544309,2.441677,294.401372,0.270397,0.184369,12.692194,2,True,12
6,NeuralNetFastAI_BAG_L2,-0.180429,-0.182412,mean_absolute_error,1.670833,2.562667,294.887783,0.396920,0.305360,13.178605,2,True,11
7,NeuralNetFastAI_BAG_L1,-0.181532,-0.188302,mean_absolute_error,0.355013,0.223930,13.337251,0.355013,0.223930,13.337251,1,True,4
8,CatBoost_BAG_L1,-0.182765,-0.180980,mean_absolute_error,0.148082,0.377707,233.379093,0.148082,0.377707,233.379093,1,True,2
9,CatBoost_r177_BAG_L1,-0.183590,-0.181814,mean_absolute_error,0.032270,0.255554,9.711147,0.032270,0.255554,9.711147,1,True,6


In [53]:
fig = px.scatter(
    x=actual_dollars,
    y=preds_dollars,
    labels={"x": "Actual Salary ($)", "y": "Predicted Salary ($)"},
    title="Cleaned + Log-Target: Actual vs Predicted",
    opacity=0.3,
)
fig.add_shape(
    type="line", x0=0, y0=0,
    x1=actual_dollars.max(), y1=actual_dollars.max(),
    line=dict(dash="dash", color="red"),
)
fig.update_layout(showlegend=False)
fig.show()

In [54]:
v2_residuals = actual_dollars - preds_dollars

fig = px.histogram(
    v2_residuals,
    nbins=80,
    title="Cleaned + Log-Target: Residuals (Actual - Predicted)",
    labels={"value": "Residual ($)"},
)
fig.add_vline(x=0, line_dash="dash", line_color="red")
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
top_countries = test_clean["Country"].value_counts().head(10).index
test_top = test_clean[test_clean["Country"].isin(top_countries)].copy()
test_top["_actual"] = 10 ** test_top[LOG_TARGET]
test_top["_pred"] = 10 ** predictor_v2.predict(test_top)
test_top["_abs_err"] = (test_top["_actual"] - test_top["_pred"]).abs()

country_mae = (
    test_top.groupby("Country")["_abs_err"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "MAE", "count": "n"})
    .sort_values("MAE")
)

fig = px.bar(
    country_mae.reset_index(),
    x="MAE",
    y="Country",
    orientation="h",
    text="n",
    title="MAE by Country (top 10 by sample size)",
    labels={"MAE": "Mean Absolute Error ($)", "n": "Samples"},
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False, yaxis={"categoryorder": "total ascending"})
fig.show()

In [56]:
predictor_v2.feature_importance(test_clean)

These features in provided data are not utilized by the predictor and will be ignored: ['MainBranch']
Computing feature importance via permutation shuffling for 38 features using 3851 rows with 5 shuffle sets...
	395.11s	= Expected runtime (79.02s per shuffle set)
	241.99s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
Country,0.148590,0.001943,3.509977e-09,5,0.152591,0.144588
WorkExp,0.023896,0.000464,1.711805e-08,5,0.024853,0.022940
OrgSize,0.009236,0.000471,8.114713e-07,5,0.010207,0.008265
YearsCode,0.005064,0.000284,1.174771e-06,5,0.005648,0.004480
Employment,0.004102,0.000420,1.295563e-05,5,0.004967,0.003238
DevType,0.003648,0.000787,2.443373e-04,5,0.005268,0.002028
RemoteWork,0.003097,0.000139,4.900829e-07,5,0.003384,0.002810
Industry,0.002845,0.000290,1.285261e-05,5,0.003443,0.002247
Age,0.002309,0.000312,3.882832e-05,5,0.002951,0.001668
EdLevel,0.002169,0.000251,2.121058e-05,5,0.002686,0.001652


### Baseline vs Cleaned + Log-Target: Summary

In [57]:
comparison = pd.DataFrame({
    "Metric": ["MAE ($)", "Median AE ($)", "RMSE ($)", "R²"],
    "Baseline": [
        f"${mean_absolute_error(baseline_actual, baseline_preds):,.0f}",
        f"${np.median(np.abs(baseline_actual - baseline_preds)):,.0f}",
        f"${root_mean_squared_error(baseline_actual, baseline_preds):,.0f}",
        f"{r2_score(baseline_actual, baseline_preds):.4f}",
    ],
    "Cleaned + Log-Target": [
        f"${dollar_mae:,.0f}",
        f"${dollar_median_ae:,.0f}",
        f"${dollar_rmse:,.0f}",
        f"{dollar_r2:.4f}",
    ],
})
comparison.style.hide(axis="index")

Metric,Baseline,Cleaned + Log-Target
MAE ($),"$40,999","$28,671"
Median AE ($),"$17,195","$16,564"
RMSE ($),"$231,776","$47,826"
R²,0.0495,0.5524
